### Imports & Downloads

In [155]:
%pip install uv --quiet
%uv pip install pandas numpy scipy plotly matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.9 environment at: c:\Users\matt\AppData\Local\Programs\Python\Python313
Checked 5 packages in 12ms


In [156]:
# Data manipulation tools
import pandas as pd
import numpy as np
from scipy import stats

# Visualization tools
import plotly.express as px
import plotly.graph_objects as go

# OS tools
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

### Downloads

In [157]:
stooq_tickers = {
    "Crypto ETFs": [
        "BITW",  # Bitwise 10 Crypto Index
        "IBIT",  # iShares Bitcoin Trust
        "ETHA",  # iShares Ethereum Trust
    ],

    "Individual Stocks": [
        "NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"
    ],

    "Sector ETFs": [
        "XLU"    # Utilities Select Sector SPDR Fund
    ],

    "Broad Market ETFs": [
        "SPY",   # S&P 500
        "VTI",   # Total US Market
    ],

    "Commodity ETFs (Metals)": [
        "GLD",   # Gold (Baseline)
        "SLV",   # Silver
        "PPLT",  # Platinum
        "PALL"   # Palladium
    ],

    "Commodity ETFs (Agriculture)": [
        "WEAT",  # Wheat
        "SOYB",  # Soybeans
        "DBA"    # Broad Agriculture
    ]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [158]:
# -----------------------------
# Flatten tickers + category map
# -----------------------------
category_map = {
    ticker: category
    for category, tickers in stooq_tickers.items()
    for ticker in tickers
}

flat_tickers = list(category_map.keys())


# -----------------------------
# Download data
# -----------------------------
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:

    valid_tickers = [t for t in flat_tickers if processor.has_ticker(t)]

    missing = sorted(set(flat_tickers) - set(valid_tickers))
    if missing:
        print("Skipping missing tickers:", missing)

    data = processor.download(
        valid_tickers,
        start=start_date,
        end=end_date,
    )


# -----------------------------
# Attach metadata
# -----------------------------
for ticker, frame in data.items():
    data[ticker] = frame.assign(
        Ticker=ticker,
        Category=category_map[ticker],
    )


# -----------------------------
# Combine dataset
# -----------------------------
combined_data = pd.concat(data.values()).reset_index()

print(combined_data)

           Date     Open     High      Low    Close       Volume  OpenInt  \
0    2025-12-31  62.8200  63.1200  56.5600  58.7600    1752680.0        0   
1    2026-01-31  59.9400  66.4800  54.5000  55.6601    2425192.0        0   
2    2026-02-28  51.2878  52.2950  40.6599  43.0400    4596906.0        0   
3    2026-03-26  42.9952  49.4500  42.9952  44.8700    1583267.0        0   
4    2024-01-31  26.4000  26.4100  22.0200  24.3000  207876989.0        0   
...         ...      ...      ...      ...      ...          ...      ...   
1307 2025-11-30  26.6000  26.9400  25.5500  26.4200    3917732.0        0   
1308 2025-12-31  26.3600  26.6666  25.4000  25.5200    4877035.0        0   
1309 2026-01-31  25.5000  26.0500  25.4250  25.6600    5404587.0        0   
1310 2026-02-28  25.5500  26.1500  25.5400  26.0200    5422359.0        0   
1311 2026-03-26  26.0400  27.1429  25.8600  27.1100   37145895.0        0   

     Ticker                      Category  
0      BITW                   C

### Correlation Analysis: Market Correlation During Stress Periods

A good store of value should move independently from the broad market - especially during downturns. This section measures each asset's correlation to VTI (total US market) across normal and stress regimes to identify which assets hold their diversification properties when it matters most.

In [159]:
# Pivot so each column is one ticker and each row is a date.
# Then compute period-over-period percentage returns.
# We use returns (not raw prices) because correlation on prices is misleading -
# two assets can trend upward together just from inflation and look correlated
# even if they're fundamentally unrelated.
# dropna() removes the first row (no return calculable) and any tickers with gaps.
pivot = combined_data.pivot_table(index="Date", columns="Ticker", values="Close")
returns = pivot.pct_change().dropna(how="all")

# VTI (Vanguard Total Stock Market ETF) is our broad market benchmark.
# Every other asset will be measured against this.
BENCHMARK = "VTI"
other_tickers = [t for t in returns.columns if t != BENCHMARK]

print(f"{len(returns.columns)} tickers, {len(returns)} observations")
print(f"Date range: {returns.index[0]} â†’ {returns.index[-1]}")
print(f"All dates:\n{returns.index.tolist()}")
returns.tail()

23 tickers, 62 observations
Date range: 2021-02-28 00:00:00 â†’ 2026-03-26 00:00:00
All dates:
[Timestamp('2021-02-28 00:00:00'), Timestamp('2021-03-31 00:00:00'), Timestamp('2021-04-30 00:00:00'), Timestamp('2021-05-31 00:00:00'), Timestamp('2021-06-30 00:00:00'), Timestamp('2021-07-31 00:00:00'), Timestamp('2021-08-31 00:00:00'), Timestamp('2021-09-30 00:00:00'), Timestamp('2021-10-31 00:00:00'), Timestamp('2021-11-30 00:00:00'), Timestamp('2021-12-31 00:00:00'), Timestamp('2022-01-31 00:00:00'), Timestamp('2022-02-28 00:00:00'), Timestamp('2022-03-31 00:00:00'), Timestamp('2022-04-30 00:00:00'), Timestamp('2022-05-31 00:00:00'), Timestamp('2022-06-30 00:00:00'), Timestamp('2022-07-31 00:00:00'), Timestamp('2022-08-31 00:00:00'), Timestamp('2022-09-30 00:00:00'), Timestamp('2022-10-31 00:00:00'), Timestamp('2022-11-30 00:00:00'), Timestamp('2022-12-31 00:00:00'), Timestamp('2023-01-31 00:00:00'), Timestamp('2023-02-28 00:00:00'), Timestamp('2023-03-31 00:00:00'), Timestamp('2023-04-3

Ticker,AAPL,AMD,AMZN,BITW,DBA,ETHA,GLD,HD,IBIT,JNJ,...,PALL,PPLT,SLV,SOYB,SPY,TSLA,VTI,WEAT,WMT,XLU
Date,,,,,,,,,,,,,,,,,,,,,
2025-11-30,0.032365,-0.150672,-0.045041,NaN,0.001896,-0.218324,0.053678,-0.059722,-0.172552,0.095568,...,0.005772,0.063049,0.163599,0.019879,0.001950,-0.057802,0.002653,-0.016981,0.092212,0.017172
2025-12-31,-0.025067,-0.015492,-0.010291,NaN,-0.034065,-0.022658,0.021734,-0.035918,-0.036857,0.000145,...,0.097788,0.221771,0.257957,-0.073729,-0.002151,0.045447,-0.003092,-0.041747,0.008144,-0.050865
2026-01-31,-0.045538,0.105388,0.036739,-0.052755,0.005486,-0.100758,0.122732,0.088608,-0.043505,0.098091,...,0.058330,0.046184,0.171065,0.017383,0.014738,-0.042938,0.015808,0.050075,0.069383,0.013118
2026-02-28,0.018113,-0.154269,-0.122440,-0.226735,0.014030,-0.280119,0.087201,0.016365,-0.216888,0.093201,...,0.055440,0.100954,0.126591,0.071043,-0.008642,-0.064822,-0.005285,0.076299,0.073947,0.103584
2026-03-26,-0.042736,0.017781,-0.011714,0.042519,0.041891,0.064738,-0.171804,-0.137345,0.043829,-0.036992,...,-0.246382,-0.229451,-0.284975,0.022670,-0.059622,-0.075526,-0.056735,0.024369,-0.045096,-0.050283


In [160]:
# We split the full timeline into market regimes to test whether an asset's
# diversification holds up specifically when the market is under stress.
# An asset that looks uncorrelated during calm periods but crashes alongside
# everything else in a downturn provides no real protection.
#
# Stress periods chosen based on documented market events:
#   - 2022 Bear Market: aggressive Fed rate hikes drove SPY down ~25%
#   - 2025 Tariff Shock: Trump tariff escalation caused broad market selloff

stress_periods = {
    "2022 Bear Market":  ("2022-01-01", "2022-10-31"),
    "2025 Tariff Shock": ("2025-02-01", "2025-04-30"),
}

normal_mask = pd.Series(True, index=returns.index)
for start, end in stress_periods.values():
    normal_mask &= ~((returns.index >= start) & (returns.index <= end))

regimes = {
    "Full Period":       returns,
    "2022 Bear Market":  returns["2022-01-01":"2022-10-31"],
    "2025 Tariff Shock": returns["2025-02-01":"2025-04-30"],
}

# Normal returns still needed for the shift table downstream
normal_returns = returns[normal_mask]

for name, df in regimes.items():
    print(f"{name}: {len(df)} observations")
    if len(df) > 0:
        print(f"  dates: {df.index.tolist()}")
    else:
        print(f"  â†’ no dates matched, index sample: {returns.index[:5].tolist()}")

Full Period: 62 observations
  dates: [Timestamp('2021-02-28 00:00:00'), Timestamp('2021-03-31 00:00:00'), Timestamp('2021-04-30 00:00:00'), Timestamp('2021-05-31 00:00:00'), Timestamp('2021-06-30 00:00:00'), Timestamp('2021-07-31 00:00:00'), Timestamp('2021-08-31 00:00:00'), Timestamp('2021-09-30 00:00:00'), Timestamp('2021-10-31 00:00:00'), Timestamp('2021-11-30 00:00:00'), Timestamp('2021-12-31 00:00:00'), Timestamp('2022-01-31 00:00:00'), Timestamp('2022-02-28 00:00:00'), Timestamp('2022-03-31 00:00:00'), Timestamp('2022-04-30 00:00:00'), Timestamp('2022-05-31 00:00:00'), Timestamp('2022-06-30 00:00:00'), Timestamp('2022-07-31 00:00:00'), Timestamp('2022-08-31 00:00:00'), Timestamp('2022-09-30 00:00:00'), Timestamp('2022-10-31 00:00:00'), Timestamp('2022-11-30 00:00:00'), Timestamp('2022-12-31 00:00:00'), Timestamp('2023-01-31 00:00:00'), Timestamp('2023-02-28 00:00:00'), Timestamp('2023-03-31 00:00:00'), Timestamp('2023-04-30 00:00:00'), Timestamp('2023-05-31 00:00:00'), Timestamp

In [161]:
# For each asset, compute its Pearson correlation to VTI within a given regime.
# Pearson r ranges from -1 to 1:
# p_value tells us whether the correlation is statistically meaningful.
# With monthly data, short stress windows may not produce significant results -
# those will show as grey in the charts below.
# Tickers with fewer than 3 shared observations are skipped entirely.

def corr_to_benchmark(regime_returns, benchmark, tickers, min_obs=3):
    rows = []
    bench = regime_returns[benchmark].dropna()

    for ticker in tickers:
        asset = regime_returns[ticker].dropna()
        shared = bench.index.intersection(asset.index)
        if len(shared) < min_obs:
            continue
        # pearsonr returns NaN if either series is constant (zero variance)
        if asset.loc[shared].std() == 0 or bench.loc[shared].std() == 0:
            continue
        r, p = stats.pearsonr(bench.loc[shared], asset.loc[shared])
        rows.append({"Ticker": ticker, "Correlation": r, "p_value": p, "n_obs": len(shared)})

    if not rows:
        return pd.DataFrame(columns=["Ticker", "Correlation", "p_value", "n_obs"])

    return pd.DataFrame(rows).sort_values("Correlation")


# Run for all regimes
regime_corrs = {
    name: corr_to_benchmark(df, BENCHMARK, other_tickers)
    for name, df in regimes.items()
}

for name, df in regime_corrs.items():
    print(f"{name}: {len(df)} tickers, obs range: {df['n_obs'].min() if not df.empty else 'N/A'}-{df['n_obs'].max() if not df.empty else 'N/A'}")

regime_corrs["Full Period"]

Full Period: 22 tickers, obs range: 3-62
2022 Bear Market: 19 tickers, obs range: 10-10
2025 Tariff Shock: 21 tickers, obs range: 3-3


,Ticker,Correlation,p_value,n_obs
3,BITW,-0.559234,6.221908e-01,3
19,WEAT,0.030976,8.111050e-01,62
16,SOYB,0.082668,5.229667e-01,62
13,PALL,0.106394,4.104871e-01,62
6,GLD,0.178550,1.649912e-01,62
4,DBA,0.203802,1.121048e-01,62
14,PPLT,0.241046,5.911992e-02,62
15,SLV,0.251898,4.826219e-02,62
9,JNJ,0.291092,2.171216e-02,62
18,TSLA,0.492660,4.749268e-05,62


In [162]:
# Bar chart of each asset's correlation to VTI for a given regime.
# Assets are sorted low to high (best diversifiers on the left).
# Blue = statistically significant (p < 0.05), grey = not significant.
# The dashed line at 0 is the ideal: no relationship to the market.

def plot_market_corr(corr_df, regime_name, benchmark=BENCHMARK):
    if corr_df.empty:
        print(f"Skipping '{regime_name}': no tickers with sufficient data.")
        return

    df = corr_df.copy()
    df["Significant"] = df["p_value"] < 0.05

    fig = px.bar(
        df,
        x="Ticker",
        y="Correlation",
        color="Significant",
        color_discrete_map={True: "steelblue", False: "lightgrey"},
        hover_data={"n_obs": True, "p_value": ":.3f"},
        title=f"Correlation to {benchmark} - {regime_name}",
        labels={"Correlation": f"Pearson r vs {benchmark}"},
        category_orders={"Ticker": df["Ticker"].tolist()},
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.update_layout(
        template="plotly_white",
        showlegend=True,
        legend_title_text="p < 0.05",
        xaxis_tickangle=-45,
        yaxis=dict(range=[-1, 1]),
    )
    fig.show()


for regime_name, corr_df in regime_corrs.items():
    plot_market_corr(corr_df, regime_name)

In [163]:
normal_corr = corr_to_benchmark(normal_returns, BENCHMARK, other_tickers).set_index("Ticker")[["Correlation"]].rename(columns={"Correlation": "Normal"})
stress_2022  = regime_corrs["2022 Bear Market"].set_index("Ticker")[["Correlation"]].rename(columns={"Correlation": "2022 Bear"})
stress_2025  = regime_corrs["2025 Tariff Shock"].set_index("Ticker")[["Correlation"]].rename(columns={"Correlation": "2025 Tariff"})

shift = normal_corr.join(stress_2022, how="outer").join(stress_2025, how="outer")
shift["d 2022"]  = shift["2022 Bear"]  - shift["Normal"]
shift["d 2025"]  = shift["2025 Tariff"] - shift["Normal"]
shift = shift.sort_values("d 2022", ascending=False)

print(shift.round(3))

        Normal  2022 Bear  2025 Tariff  d 2022  d 2025
Ticker                                                
AAPL     0.470      0.934        0.628   0.464   0.158
NVDA     0.497      0.946        0.922   0.449   0.425
AMD      0.375      0.710       -0.736   0.335  -1.111
MSFT     0.604      0.910        0.730   0.307   0.126
XLU      0.505      0.788        0.223   0.283  -0.282
AMZN     0.543      0.731        0.636   0.188   0.093
TSLA     0.442      0.595        0.268   0.153  -0.174
WMT      0.445      0.537        0.958   0.092   0.513
DBA      0.142      0.229        0.093   0.087  -0.050
PPLT     0.253      0.317       -0.966   0.064  -1.219
LOW      0.619      0.633        0.995   0.014   0.377
SPY      0.992      0.998        0.992   0.006   0.000
HD       0.671      0.656        0.988  -0.015   0.317
SLV      0.263      0.242       -0.998  -0.020  -1.261
JNJ      0.352      0.306       -0.132  -0.046  -0.484
WEAT     0.129     -0.024        0.008  -0.153  -0.121
GLD      0

### Visualization: Correlation Shift Under Stress

In [164]:
# Grouped bar chart of the correlation shift for both stress periods.
# Assets sorted by their 2022 shift (worst offenders on the left).
# Bars above 0 = diversification failed under that stress event.
# Bars below 0 = asset actually decoupled from the market under stress.

print("shift columns:", shift.columns.tolist())
print("shift shape:", shift.shape)
print(shift.round(3))


shift columns: ['Normal', '2022 Bear', '2025 Tariff', 'd 2022', 'd 2025']
shift shape: (22, 5)
        Normal  2022 Bear  2025 Tariff  d 2022  d 2025
Ticker                                                
AAPL     0.470      0.934        0.628   0.464   0.158
NVDA     0.497      0.946        0.922   0.449   0.425
AMD      0.375      0.710       -0.736   0.335  -1.111
MSFT     0.604      0.910        0.730   0.307   0.126
XLU      0.505      0.788        0.223   0.283  -0.282
AMZN     0.543      0.731        0.636   0.188   0.093
TSLA     0.442      0.595        0.268   0.153  -0.174
WMT      0.445      0.537        0.958   0.092   0.513
DBA      0.142      0.229        0.093   0.087  -0.050
PPLT     0.253      0.317       -0.966   0.064  -1.219
LOW      0.619      0.633        0.995   0.014   0.377
SPY      0.992      0.998        0.992   0.006   0.000
HD       0.671      0.656        0.988  -0.015   0.317
SLV      0.263      0.242       -0.998  -0.020  -1.261
JNJ      0.352      0.306

### Rolling Correlation Over Time (12-Month)

Each line shows how an asset's correlation to VTI has shifted month by month, calculated over a trailing 12-month window. Rather than a single number per regime, this reveals the full timeline - you can see correlations spike during the red-shaded stress periods or drift lower during calm stretches. A line that stays consistently near 0 across the entire chart, including inside the stress periods, indicates an asset that is structurally independent from the market over time. Lines that spike upward into the red zones are assets whose diversification broke down exactly when it was needed most. Note: lines only appear once 12 months of data have accumulated, which is why they start partway into the chart.

In [165]:
ROLLING_WINDOW = 11  # months

rolling_corr = (
    returns[other_tickers]
    .apply(lambda col: col.rolling(ROLLING_WINDOW).corr(returns[BENCHMARK]))
)

fig = px.line(
    rolling_corr.reset_index().melt(id_vars="Date", var_name="Ticker", value_name="Rolling Correlation"),
    x="Date",
    y="Rolling Correlation",
    color="Ticker",
    title=f"12-Month Rolling Correlation to {BENCHMARK}",
    labels={"Rolling Correlation": "Pearson r (12-month window)"},
)

# Shade stress periods
for name, (start, end) in stress_periods.items():
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor="red", opacity=0.08,
        line_width=0,
        annotation_text=name,
        annotation_position="top left",
    )

fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", yaxis=dict(range=[-1, 1]))
fig.show()

### Beta vs. VTI

Correlation measures *direction* but not *magnitude*. Beta answers: when VTI drops 1%, how much does this asset move?

- Beta < 0  moves opposite the market
- Beta 0-0.5 low sensitivity, defensive
- Beta ~ 1 moves in line with the market
- Beta > 1 amplifies market moves (high risk)

A good store of value wants low correlation *and* low beta. An asset with high correlation but very low beta is still relatively defensive in practice.

In [166]:
vti = returns[BENCHMARK]
vti_var = vti.var()

beta_rows = []
for ticker in other_tickers:
    asset = returns[ticker].dropna()
    shared = vti.index.intersection(asset.index)
    if len(shared) < 3:
        continue
    cov = vti.loc[shared].cov(asset.loc[shared])
    beta_rows.append({"Ticker": ticker, "Beta": cov / vti_var})

beta_df = pd.DataFrame(beta_rows).sort_values("Beta")

# Merge with full-period correlation for a combined view
full_corr = regime_corrs["Full Period"].set_index("Ticker")[["Correlation"]]
beta_df = beta_df.set_index("Ticker").join(full_corr).reset_index()
beta_df = beta_df.sort_values("Beta")

fig = px.bar(
    beta_df,
    x="Ticker",
    y="Beta",
    title=f"Beta vs {BENCHMARK} - Full Period<br><sup>How much does each asset move per 1% move in VTI?</sup>",
    labels={"Beta": "Beta vs VTI"},
    category_orders={"Ticker": beta_df["Ticker"].tolist()},
    color="Beta",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
)
fig.add_hline(y=1, line_dash="dash", line_color="black", annotation_text="Beta = 1 (moves with market)")
fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

beta_df.sort_values("Beta")

,Ticker,Beta,Correlation
0,BITW,-1.439262,-0.559234
1,WEAT,0.047136,0.030976
2,SOYB,0.084118,0.082668
3,DBA,0.154307,0.203802
4,GLD,0.200628,0.178550
5,PALL,0.231425,0.106394
6,JNJ,0.317700,0.291092
7,PPLT,0.426896,0.241046
8,SLV,0.513797,0.251898
9,WMT,0.640378,0.501238


In [167]:
# Beta vs VTI computed using only the 2025 Tariff Shock window (Feb-Apr 2025)
# Shows how sensitive each asset was to market moves during that specific event
tariff_returns = returns['2025-02-01':'2025-04-30']
tariff_vti = tariff_returns[BENCHMARK]
tariff_vti_var = tariff_vti.var()

tariff_beta_rows = []
for ticker in other_tickers:
    asset = tariff_returns[ticker].dropna()
    shared = tariff_vti.index.intersection(asset.index)
    if len(shared) < 3:
        continue
    cov = tariff_vti.loc[shared].cov(asset.loc[shared])
    tariff_beta_rows.append({'Ticker': ticker, 'Beta': cov / tariff_vti_var})

if not tariff_beta_rows:
    print('Not enough observations in the 2025 Tariff window to compute beta.')
else:
    tariff_beta_df = pd.DataFrame(tariff_beta_rows).sort_values('Beta')

    fig = px.bar(
        tariff_beta_df,
        x='Ticker',
        y='Beta',
        title=f'Beta vs {BENCHMARK} - 2025 Tariff Shock',
        labels={'Beta': 'Beta vs VTI'},
        category_orders={'Ticker': tariff_beta_df['Ticker'].tolist()},
        color='Beta',
        color_continuous_scale='RdBu_r',
        color_continuous_midpoint=0,
    )
    fig.add_hline(y=1, line_dash='dash', line_color='black', annotation_text='Beta = 1 (moves with market)')
    fig.add_hline(y=0, line_dash='dot', line_color='grey')
    fig.update_layout(template='plotly_white', xaxis_tickangle=-45)
    fig.show()

    print(tariff_beta_df.sort_values('Beta'))

   Ticker      Beta
12   PALL -2.625111
14    SLV -2.507204
1     AMD -2.148998
13   PPLT -1.796163
5     GLD -1.020797
8     JNJ -0.327545
18   WEAT  0.001468
20    XLU  0.070653
15   SOYB  0.074560
3     DBA  0.094246
9     LOW  0.388905
16    SPY  0.900939
2    AMZN  0.954614
6      HD  1.039566
4    ETHA  1.088281
0    AAPL  1.190667
7    IBIT  1.278632
10   MSFT  1.501291
17   TSLA  1.705727
11   NVDA  2.930289
19    WMT  3.627644


### Downside Capture

Splits all periods into two buckets: months where VTI was down vs. months where VTI was up. For each asset, computes the average return in each bucket.

- **Low downside capture** (small negative bar when market drops) â†’ asset resists market declines
- **Reasonable upside capture** (positive bar when market rises) â†’ asset still grows over time

This is the most intuitive read for a non-technical audience - it directly answers "what happens to this asset when the market crashes?"

In [168]:
vti_down = returns[BENCHMARK] < 0
vti_up   = returns[BENCHMARK] >= 0

capture_rows = []
for ticker in other_tickers:
    asset = returns[ticker].dropna()
    shared_down = vti_down.index.intersection(asset.index)
    shared_up   = vti_up.index.intersection(asset.index)
    capture_rows.append({
        "Ticker": ticker,
        "Avg Return (VTI Down)": asset.loc[shared_down[vti_down.loc[shared_down]]].mean(),
        "Avg Return (VTI Up)":   asset.loc[shared_up[vti_up.loc[shared_up]]].mean(),
    })

capture_df = pd.DataFrame(capture_rows)
capture_df = capture_df.sort_values("Avg Return (VTI Down)", ascending=False)  # best defenders first

capture_melted = capture_df.melt(id_vars="Ticker", var_name="Market Condition", value_name="Avg Return")

fig = px.bar(
    capture_melted,
    x="Ticker",
    y="Avg Return",
    color="Market Condition",
    barmode="group",
    title=f"Downside & Upside Capture vs {BENCHMARK}<br><sup>Average asset return when VTI is down (red) vs up (green)</sup>",
    labels={"Avg Return": "Average Period Return"},
    color_discrete_map={
        "Avg Return (VTI Down)": "crimson",
        "Avg Return (VTI Up)":   "seagreen",
    },
    category_orders={"Ticker": capture_df["Ticker"].tolist()},
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

capture_df.sort_values("Avg Return (VTI Down)", ascending=False)

,Ticker,Avg Return (VTI Down),Avg Return (VTI Up)
6,GLD,0.001297,0.022941
16,SOYB,0.000290,0.006270
19,WEAT,-0.000632,-0.003973
4,DBA,-0.002030,0.015029
14,PPLT,-0.006146,0.021233
15,SLV,-0.010623,0.035680
20,WMT,-0.018102,0.039194
9,JNJ,-0.019595,0.024541
21,XLU,-0.022007,0.028548
13,PALL,-0.029230,0.011138


### Conditional Correlation - Full Period vs. Worst 20% of Months

The full period correlation includes all market conditions, which can mask how an asset behaves specifically during the worst drawdowns. This chart isolates only the bottom 20% of VTI return months - the most extreme market drops in the dataset - and recomputes correlation there. If an asset's bar is higher in the tail than in the full period, its correlation to the market spiked during the worst months, meaning it sold off alongside equities precisely when a store of value should have held firm. Assets whose tail bar stays flat or drops below the full period bar are the most robust candidates - they maintained or strengthened their independence even under the most severe market conditions.

In [169]:
TAIL_PCT = 0.20  # bottom 20% of VTI months

tail_threshold = returns[BENCHMARK].quantile(TAIL_PCT)
tail_dates = returns.index[returns[BENCHMARK] <= tail_threshold]
tail_returns = returns.loc[tail_dates]

print(f"Tail threshold (bottom {int(TAIL_PCT*100)}%): VTI return <= {tail_threshold:.3f}")
print(f"Tail observations: {len(tail_dates)}")

tail_corrs = corr_to_benchmark(tail_returns, BENCHMARK, other_tickers)

# Compare tail correlation vs full-period correlation
full_corr = regime_corrs["Full Period"].set_index("Ticker")[["Correlation"]].rename(columns={"Correlation": "Full Period"})
tail_corr_col = tail_corrs.set_index("Ticker")[["Correlation"]].rename(columns={"Correlation": "Tail (Worst 20%)"})
comparison = full_corr.join(tail_corr_col, how="outer")
comparison["Î” Tail"] = comparison["Tail (Worst 20%)"] - comparison["Full Period"]
comparison = comparison.sort_values("Î” Tail", ascending=False)

fig = px.bar(
    comparison.reset_index(),
    x="Ticker",
    y=["Full Period", "Tail (Worst 20%)"],
    barmode="group",
    title=f"Correlation to {BENCHMARK}: Full Period vs Worst 20% of Months<br><sup>Assets that spike in the tail failed exactly when it mattered most</sup>",
    labels={"value": "Pearson r", "variable": "Regime"},
    category_orders={"Ticker": comparison.index.tolist()},
)
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45, yaxis=dict(range=[-1, 1]))
fig.show()

comparison.round(3)

Tail threshold (bottom 20%): VTI return <= -0.026
Tail observations: 13


,Full Period,Tail (Worst 20%),Î” Tail
Ticker,,,
DBA,0.204,0.474,0.270
AMZN,0.664,0.829,0.165
AMD,0.500,0.615,0.115
NVDA,0.680,0.770,0.089
AAPL,0.655,0.738,0.084
WEAT,0.031,0.083,0.052
MSFT,0.684,0.697,0.012
SPY,0.995,0.983,-0.013
GLD,0.179,0.155,-0.023


### Cumulative Returns (Normalized to 100)

Every asset starts at 100 at the beginning of the data window. The chart then shows how $100 invested in each asset grew (or shrunk) over the full period.

This is the most direct long-term view - it cuts through month-to-month noise and answers the core question: did this asset actually preserve and grow wealth over time? An asset that looks volatile in the correlation charts might still compound well over years, and vice versa.

In [170]:
# Normalize each asset's price to 100 at the start of the data window.
# cumulative return = (price / first_price) * 100
# This removes the nominal price difference between assets (GLD at $180 vs AGG at $95)
# and puts everyone on the same footing - pure % growth from the same starting point.

first_prices = pivot.iloc[0]
cumulative = (pivot / first_prices) * 1

cum_melted = cumulative.reset_index().melt(id_vars="Date", var_name="Ticker", value_name="Cumulative Return")

fig = px.line(
    cum_melted,
    x="Date",
    y="Cumulative Return",
    color="Ticker",
    title="Cumulative Returns Testing",
    labels={"Cumulative Return": "Value of $1 invested"},
)

# Shade stress periods
for name, (start, end) in stress_periods.items():
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor="red", opacity=0.08,
        line_width=0,
        annotation_text=name,
        annotation_position="top left",
    )

# Reference line at 100 = no gain/loss from starting point
fig.add_hline(y=100, line_dash="dash", line_color="black", annotation_text="Break even")

fig.update_layout(
    template="plotly_white",
    height=600,
    yaxis_title="Value of $1 invested",
)
fig.show()

In [171]:
# Export correlation analysis results to CSV for use in the report and cross-notebook comparison.
# Output: data/correlation_metrics.csv

output_path = repo_root / 'data' / 'correlation_metrics.csv'

# Full period correlation and beta
export_df = regime_corrs['Full Period'].set_index('Ticker')[['Correlation', 'p_value', 'n_obs']].rename(columns={
    'Correlation': 'Full_Period_Corr',
    'p_value':     'Full_Period_PVal',
    'n_obs':       'Full_Period_Obs',
})

# Add 2022 and 2025 stress correlations
for regime_name, col_prefix in [('2022 Bear Market', 'Bear2022'), ('2025 Tariff Shock', 'Tariff2025')]:
    df = regime_corrs[regime_name]
    if not df.empty:
        export_df = export_df.join(
            df.set_index('Ticker')[['Correlation', 'p_value']].rename(columns={
                'Correlation': f'{col_prefix}_Corr',
                'p_value':     f'{col_prefix}_PVal',
            }),
            how='outer'
        )

# Add beta
export_df = export_df.join(
    beta_df.set_index('Ticker')[['Beta']],
    how='outer'
)

# Add downside/upside capture
export_df = export_df.join(
    capture_df.set_index('Ticker'),
    how='outer'
)

export_df = export_df.round(4)
export_df.to_csv(output_path)
print(f'Exported {len(export_df)} tickers to {output_path}')
export_df

Exported 22 tickers to c:\Users\matt\Documents\Coding\Modern Store\Modern-Store-Of-Value\data\correlation_metrics.csv


,Full_Period_Corr,Full_Period_PVal,Full_Period_Obs,Bear2022_Corr,Bear2022_PVal,Tariff2025_Corr,Tariff2025_PVal,Beta,Avg Return (VTI Down),Avg Return (VTI Up)
Ticker,,,,,,,,,,
AAPL,0.6545,0.0000,62,0.9339,0.0001,0.6283,0.5675,1.0348,-0.0340,0.0413
AMD,0.5004,0.0000,62,0.7097,0.0215,-0.7364,0.4731,1.8349,-0.0632,0.0790
AMZN,0.6639,0.0000,62,0.7315,0.0162,0.6361,0.5611,1.3373,-0.0531,0.0442
BITW,-0.5592,0.6222,3,NaN,NaN,NaN,NaN,-1.4393,-0.0921,-0.0528
DBA,0.2038,0.1121,62,0.2291,0.5243,0.0927,0.9409,0.1543,-0.0020,0.0150
ETHA,0.5276,0.0168,20,NaN,NaN,0.2017,0.8707,2.0897,-0.1089,0.0717
GLD,0.1786,0.1650,62,0.0553,0.8793,-0.7686,0.4419,0.2006,0.0013,0.0229
HD,0.6827,0.0000,62,0.6561,0.0394,0.9885,0.0968,1.0944,-0.0488,0.0389
IBIT,0.5280,0.0056,26,NaN,NaN,0.2344,0.8494,1.4476,-0.0409,0.0665


### MinMax Normalization (1–10)

Each metric is normalized to a 1–10 scale using MinMax scaling across all assets.
- Metrics where **higher is better** (e.g. inflation-adjusted return): raw value → score
- Metrics where **lower is better** (e.g. max drawdown, correlation to market): values are inverted before scaling

Formula: `score = 1 + (x - x_min) / (x_max - x_min) * 9`

Scores are decimal values — a 6.3 is a real score, not rounded.


In [ ]:
# Load source metrics, Correlation and Beta
# Both come from correlation_metrics.csv exported by this notebook (cell 25)

from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parent

corr_df = pd.read_csv(repo_root / "data" / "correlation_metrics.csv").set_index("Ticker")

metrics_raw = pd.DataFrame(index=corr_df.index)
metrics_raw["Market_Corr"] = corr_df["Full_Period_Corr"]   # abs distance from 0, lower = more independent
metrics_raw["Beta"]        = corr_df["Beta"]               # abs distance from 0, lower = more independent
metrics_raw = metrics_raw.dropna()

print("Assets being scored:", metrics_raw.index.tolist())
print(metrics_raw.round(3))


Assets being scored: ['AAPL', 'AMD', 'AMZN', 'BITW', 'DBA', 'ETHA', 'GLD', 'HD', 'IBIT', 'JNJ', 'LOW', 'MSFT', 'NVDA', 'PALL', 'PPLT', 'SLV', 'SOYB', 'SPY', 'TSLA', 'WEAT', 'WMT', 'XLU']
        Market_Corr   Beta
Ticker                    
AAPL          0.654  1.035
AMD           0.500  1.835
AMZN          0.664  1.337
BITW         -0.559 -1.439
DBA           0.204  0.154
ETHA          0.528  2.090
GLD           0.179  0.201
HD            0.683  1.094
IBIT          0.528  1.448
JNJ           0.291  0.318
LOW           0.630  1.005
MSFT          0.684  1.034
NVDA          0.680  2.222
PALL          0.106  0.231
PPLT          0.241  0.427
SLV           0.252  0.514
SOYB          0.083  0.084
SPY           0.995  0.981
TSLA          0.493  1.888
WEAT          0.031  0.047
WMT           0.501  0.640
XLU           0.574  0.641


In [ ]:
# Step 1: MinMax normalization → 1-10
# Step 2: Percentile normalization applied to MinMax scores → 1-10

def minmax_score(series, higher_is_better=True):
    """MinMax normalization scaled to [1, 10]. Min raw value → 1, max → 10."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(5.5, index=series.index)
    normalized = (series - mn) / (mx - mn)
    if not higher_is_better:
        normalized = 1 - normalized
    return 1 + normalized * 9


def percentile_score(series):
    """Percentile rank of MinMax scores scaled to [1, 10]."""
    ranked = series.rank(pct=True)
    return 1 + ranked * 9


# Absolute distance from 0 — closest to 0 = most independent from market
corr = metrics_raw["Market_Corr"].abs()
beta = metrics_raw["Beta"].abs()

# Step 1: MinMax
minmax = pd.DataFrame(index=metrics_raw.index)
minmax["Correlation"] = minmax_score(corr, higher_is_better=False)
minmax["Beta"]        = minmax_score(beta, higher_is_better=False)

# Step 2: Percentile on top of MinMax
pct = minmax.apply(percentile_score)

print("=" * 50)
print("STEP 1 — MinMax Scores (1-10)")
print("=" * 50)
print(minmax.round(2).to_string())

print()
print("=" * 50)
print("STEP 2 — Percentile Scores of MinMax (1-10)")
print("=" * 50)
print(pct.round(2).to_string())


STEP 1 — MinMax Scores (1-10)
        Correlation   Beta
Ticker                    
AAPL           4.18   5.91
AMD            5.62   2.60
AMZN           4.09   4.66
BITW           5.07   4.24
DBA            8.39   9.56
ETHA           5.36   1.55
GLD            8.62   9.36
HD             3.92   5.67
IBIT           5.36   4.20
JNJ            7.57   8.88
LOW            4.41   6.04
MSFT           3.90   5.92
NVDA           3.94   1.00
PALL           9.30   9.24
PPLT           8.04   8.43
SLV            7.94   8.07
SOYB           9.52   9.85
SPY            1.00   6.13
TSLA           5.69   2.38
WEAT          10.00  10.00
WMT            5.61   7.54
XLU            4.93   7.54

STEP 2 — Percentile Scores of MinMax (1-10)
        Correlation   Beta
Ticker                    
AAPL           3.45   4.68
AMD            6.32   2.64
AMZN           3.05   3.86
BITW           4.68   3.45
DBA            8.36   9.18
ETHA           5.50   1.82
GLD            8.77   8.77
HD             2.23   4.27
IBIT   

### Bar Plot — Market Independence Score (Correlation to VTI)

In [174]:
import plotly.express as px

# ── Build plot dataframe from correlation MinMax scores ───────────────────────
# Higher score = more independent from VTI (closer to 0 correlation)
# GLD is the baseline — assets above the line beat gold's independence

plot_df = (
    pd.DataFrame({
        "MinMax_Score":     minmax["Correlation"],
        "Percentile_Score": pct["Correlation"],
        "Raw_Corr":         metrics_raw["Market_Corr"],
    })
    .reset_index()
    .assign(Category=lambda d: d["Ticker"].map(category_map))
    .sort_values("Percentile_Score", ascending=False)
)

plot_df["Score_Formatted"] = plot_df["Percentile_Score"].round(2).astype(str)
plot_df["MinMax_Formatted"] = plot_df["MinMax_Score"].round(2).astype(str)
plot_df["Corr_Formatted"]  = plot_df["Raw_Corr"].round(3).astype(str)

# GLD baseline score
gld_score = plot_df.loc[plot_df["Ticker"] == "GLD", "Percentile_Score"].values[0]

fig = px.bar(
    plot_df,
    x="Ticker",
    y="Percentile_Score",
    color="Category",
    labels={"Percentile_Score": "Market Independence Score (10 = Most Independent)", "Ticker": "Asset"},
    hover_data={
        "Score_Formatted":  True,
        "MinMax_Formatted": True,
        "Corr_Formatted":   True,
        "Percentile_Score": False,
        "MinMax_Score":     False,
        "Raw_Corr":         False,
        "Category":         False,
    },
    template="plotly_white",
)

fig.update_layout(
    xaxis=dict(
        categoryorder="array",
        categoryarray=plot_df["Ticker"],
    ),
    title=dict(text="Market Independence — Correlation to VTI (Score 1-10)", x=0.5),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=18),
    xaxis_tickangle=-45,
    xaxis_tickfont=dict(size=11),
    yaxis=dict(range=[0, 10.5]),
    showlegend=True,
)

# GLD baseline — the benchmark every asset is compared against
fig.add_hline(
    y=gld_score,
    line_dash="dot",
    line_color="black",
    line_width=2,
    annotation_text=f"GLD Baseline (Score: {gld_score:.2f})",
    annotation_position="top right" if gld_score < 8 else "bottom right",
)

fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "market_independence_score_poster",
        "height": 600,
        "width": 1000,
        "scale": 3,
    }
})

fig.write_html(repo_root / "plots" / "score_bar_market_independence.html")
print("Saved -> plots/score_bar_market_independence.html")


Saved -> plots/score_bar_market_independence.html


### Export — Market Independence Scores to CSV

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Export MinMax and Percentile scores to CSV
# Output: data/market_independence_scores.csv
# ─────────────────────────────────────────────────────────────────────────────

export_df = pd.DataFrame({
    "Ticker":                     metrics_raw.index,
    "Raw_Correlation":            metrics_raw["Market_Corr"].values,
    "Raw_Beta":                   metrics_raw["Beta"].values,
    "Correlation_MinMax":         minmax["Correlation"].values,
    "Beta_MinMax":                minmax["Beta"].values,
    "Correlation_Percentile":     pct["Correlation"].values,
    "Beta_Percentile":            pct["Beta"].values,
})

output_path = repo_root / "data" / "market_independence_scores.csv"
export_df.to_csv(output_path, index=False)

print(f"Exported -> {output_path}")
print(export_df.round(2).to_string(index=False))
